In [17]:
import pandas as pd
import numpy as np

In [18]:
df = pd.read_csv(
    "/home/chintan/house_price_prediction/data/house_prices.csv",
    nrows=1000
)

In [19]:
def price_to_value(values):
    try:
        value = values.lower()
        if "cr" in  value:
            v = value.replace("cr","").strip()
            return float(v)*100
        v = value.replace("lac","").strip()
        if "lac" in  value:
            return float(value.replace("lac","").strip())
        return None
    except:
        return None

def car_parking_encoding(value):
    if type(value) == str and value and ("open" in value.lower() or "covered" in value.lower()):
        return 1
    return 0

48

def intTryParse(value):
    try:
        return int(value)
    except ValueError:
        return 0
    
def balcony_encoding(value):
    if type(value) != str:
        return 0
    # print(f"{value} {type(value)} {value} Before")
    if (type(value) == float or type(value) == int):
        return value
    return intTryParse(value)

def furnishing_encoding(value):
    if type(value) != str or "unfurnished" in value.lower():
        return 0
    if "semi-furnished" in value.lower() :
        return 0.5
    if "furnished" in value.lower() :
        return 1
    return 0


def overlooking_encoding(value):
    if type(value) != str:
        return None
    l = []
    if "road" in value.lower():
        l.append("road")
    if "garden" in value.lower() or "park" in value.lower():
            l.append("garden")
    if "pool" in value.lower():
            l.append("pool")   
    return ",".join(l)

In [20]:
df = df.drop(columns=["Index","Price (in rupees)","Description","Title","Transaction","facing","Society", "Super Area","Plot Area","Dimensions", "Status", "location"])
df = df.rename(columns={'Amount(in rupees)': 'amount', 'Car Parking': 'car_parking','Carpet Area': 'area'})
df["amount"] = df["amount"].apply(price_to_value)
df["car_parking"] = df["car_parking"].apply(car_parking_encoding)
df["Balcony"] = df["Balcony"].apply(balcony_encoding)
df["Furnishing"] = df["Furnishing"].apply(furnishing_encoding)
df["Bathroom"] = df["Bathroom"].apply(intTryParse)
df["overlooking"] = df["overlooking"].apply(overlooking_encoding)
df["area"] = df["area"].apply(lambda value: intTryParse(value.split(" ")[0]) if value != None and type(value) == str  else None)
df["Floor"] = df["Floor"].apply(lambda value: intTryParse(value.split(" ")[0]) if value != None and type(value) == str  else None)
df = df.dropna(subset=["amount"])
df = df.dropna(subset=["area"])
df = df.dropna(subset=["Ownership"])
df = df.dropna(subset=["Floor"])
df = pd.get_dummies(df, columns=["overlooking","Ownership"], dtype=int)
df = df.drop(df[df["area"] > 3000].index)


### Train and Test Split

In [21]:
##
df_train = df.sample(frac=0.8,random_state=42)
df_test = df.drop(df_train.index)

### Train and Test X and Y values

In [22]:
X_train = df_train.drop("amount",axis=1)
Y_train = df_train["amount"]
X_test = df_test.drop("amount",axis=1)
Y_test = df_test["amount"]

### Feature Scalling

In [23]:
mean_of_area = X_train["area"].mean()
mean_of_floor = X_train["Floor"].mean()
mean_of_bathroom = X_train["Bathroom"].mean()
mean_of_amount = Y_train.mean()

variance_area = ((X_train["area"] - mean_of_area)**2).sum()/X_train.shape[0]
variance_floor = ((X_train["Floor"] - mean_of_floor)**2).sum()/X_train.shape[0]
variance_bathroom = ((X_train["Bathroom"] - mean_of_bathroom)**2).sum()/X_train.shape[0]
variance_amount = ((Y_train - mean_of_amount)**2).sum()/Y_train.shape[0]

standard_deviation_area = variance_area**0.5
standard_deviation_floor = variance_floor**0.5
standard_deviation_bathroom = variance_bathroom**0.5
standard_deviation_amount = variance_amount**0.5

scaled_area = (X_train["area"] - mean_of_area) / standard_deviation_area
scaled_floor = (X_train["Floor"] - mean_of_floor) / standard_deviation_floor
scaled_bathroom = (X_train["Bathroom"] - mean_of_bathroom) / standard_deviation_bathroom
scaled_amount = (Y_train - mean_of_amount) / standard_deviation_amount

X_train["area"] = scaled_area
X_train["Floor"] = scaled_floor
X_train["Bathroom"] = scaled_bathroom
Y_train = scaled_amount

In [24]:
# df.head(n=50)
# Y_train

### Remove Outliners 

In [25]:

# Q1 = df["amount"].quantile(0.30)
# Q3 = df["amount"].quantile(0.80)

# IQR = Q3 - Q1

# lower = Q1 - 1.5 * IQR
# upper = Q3 + 1.5 * IQR

# outliers = df[
#     (df["amount"] < lower) |
#     (df["amount"] > upper)
# ]



In [26]:
weights = np.zeros(X_train.shape[1])

bias = 0

learning_rate = 0.01

In [27]:

for i in range(1000):
    y_predicted = (X_train * weights).sum(axis=1) + bias
    errors = (Y_train - y_predicted).to_numpy()
    mean_squered_error = ((errors**2).sum()/y_predicted.shape[0])**0.5
    gradients = (2 / X_train.shape[0]) * (errors[:, np.newaxis] * X_train).sum(axis=0)
    bias_gradient = (2 / X_train.shape[0]) * errors.sum()
    weights = weights + learning_rate * gradients
    bias = bias + learning_rate * bias_gradient
    # print(errors[:5],"==========>> Weights",i)
    # print(mean_squered_error,"==========>> MSE",i)


In [28]:
y_test_predicted = (X_test * weights).sum(axis=1) + bias

print(y_test_predicted)


2       681.572507
18      526.191780
26      251.771312
28      808.718637
34      384.317511
          ...     
960     513.577627
963     541.973919
972     569.082273
983    1180.695003
988     621.018815
Length: 112, dtype: float64


In [29]:
# mean_of_amount = Y_test.mean()
# variance_amount = ((Y_test - mean_of_amount)**2).sum()/Y_test.shape[0]
# standard_deviation_amount = variance_amount**0.5
# scaled_test_amount = (Y_test - mean_of_amount) / standard_deviation_amount
# print(y_test_predicted[:5])
# print(scaled_test_amount[:5])

